# 1. Import Libraries

# 2. import raw data

# 3. clean data
    - Clean sales_detail dataframe
    - Clean division_summary dataframe
    - Clean customer_orders dataframe
    - Clean sales_by_item_group
    - Clean branch_tax_summary
    - Clean monthly_branch_sales
    - Clean avg_sales_menu
    - Clean attendance_logs
# Export Dataframes

### Import Libraries

In [28]:
# Import Libraries
import pandas as pd
import numpy as np
import os

### import raw data

In [137]:
# path to our files
path = r'C:\Users\Public\Hackathon\Conut bakery Scaled Data_\original data'

# Import our files

# import Sales by customer file
sales_detail = pd.read_csv(os.path.join(path, 'REP_S_00502.csv'), header=None)
               
# import Customer order summary
customer_orders = pd.read_csv(os.path.join(path, 'rep_s_00150.csv'), header=None, sep=',', on_bad_lines='skip', engine='python', encoding='utf-8')

# import Sales by item & group
sales_by_item_group = pd.read_csv(os.path.join(path, 'rep_s_00191_SMRY.csv'), header=None, sep=',', on_bad_lines='skip', engine='python', encoding='utf-8')

# import Tax summary by branch
branch_tax_summary = pd.read_csv(os.path.join(path, 'REP_S_00194_SMRY.csv'), header=None, sep=',', on_bad_lines='skip', engine='python', encoding='utf-8')

# import Monthly sales by branch
monthly_branch_sales = pd.read_csv(os.path.join(path, 'rep_s_00334_1_SMRY.csv'), header=None, sep=',', on_bad_lines='skip', engine='python', encoding='utf-8')

# import Average sales by menu
avg_sales_menu = pd.read_csv(os.path.join(path, 'rep_s_00435_SMRY.csv'), header=None, sep=',', on_bad_lines='skip', engine='python', encoding='utf-8')

# import Duplicate version!!! duplicatename
avg_sales_menu_v2 = pd.read_csv(os.path.join(path, 'rep_s_00435_SMRY (1).csv'), header=None, sep=',', on_bad_lines='skip', engine='python', encoding='utf-8')

# import Time & attendance logs
attendance_logs = pd.read_csv(os.path.join(path, 'REP_S_00461.csv'), header=None, sep=',', on_bad_lines='skip', engine='python', encoding='utf-8')

# import Division / channel summary
division_summary = pd.read_csv(os.path.join(path, 'REP_S_00136_SMRY.csv'), header=None, sep=',', on_bad_lines='skip', engine='python', encoding='utf-8')


In [38]:
print("Sales detail shape:", sales_detail.shape)
print("Customer orders shape:", customer_orders.shape)
print("Sales by item group shape:", sales_by_item_group.shape)
print("Branch tax summary shape:", branch_tax_summary.shape)
print("Monthly branch sales shape:", monthly_branch_sales.shape)
print("Attendance logs shape:", attendance_logs.shape)

Sales detail shape: (2308, 5)
Customer orders shape: (543, 10)
Sales by item group shape: (1740, 5)
Branch tax summary shape: (14, 10)
Monthly branch sales shape: (40, 5)
Attendance logs shape: (398, 6)


### Clean sales_detail dataframe

In [79]:
def clean_conut_sales_df(df_input):
    clean_data = []
    current_person = None
    current_branch = "Unknown"

    for _, row in df_input.iterrows():
        col0 = str(row[0]).strip() if pd.notna(row[0]) else ""
        col1 = row[1] 
        col2 = str(row[2]).strip() if pd.notna(row[2]) else ""
        col3 = str(row[3]).strip() if pd.notna(row[3]) else "0"
      
        if "Branch :" in col0:
            current_branch = col0.split("Branch :")[-1].strip()
            continue
            
        if col0.startswith("Person_"):
            current_person = col0
            continue

        try:
            qty = float(col1)
            is_valid_qty = not np.isnan(qty)
        except:
            is_valid_qty = False

        if col0 == "" and is_valid_qty and col2 != "":
            if "Description" in col2 or "Total" in col2:
                continue

            try:
                price_val = float(col3.replace(',', '').replace('"', ''))
            except:
                price_val = 0.0

            clean_data.append({
                'Branch': current_branch,
                'Customer': current_person,
                'Qty': qty,
                'Item': col2,
                'Price': price_val
            })

    df_result = pd.DataFrame(clean_data)
    
    df_result = df_result[df_result['Qty'] > 0]
    
    return df_result

sales_detail_clean = clean_conut_sales_df(sales_detail)


In [80]:
sales_detail_clean.head()

,Branch,Customer,Qty,Item,Price
1,Conut - Tyre,Person_0129,1.0,FULL FAT MILK,0.00
2,Conut - Tyre,Person_0129,1.0,PISTACHIO MILKSHAKE,893918.92
5,Conut - Tyre,Person_0129,1.0,WHIPPED CREAM...,0.00
6,Conut - Tyre,Person_0130,1.0,"CARAMEL SAUCE, (R)",0.00
7,Conut - Tyre,Person_0130,1.0,CHIMNEY THE ONE,1251486.48


In [81]:
sales_detail_clean.tail()

,Branch,Customer,Qty,Item,Price
1876,Main Street Coffee,Person_0249,1.0,CHIMNEY THE ONE,1251486.48
1877,Main Street Coffee,Person_0249,1.0,CRUSHED LOTUS .(R),0.00
1878,Main Street Coffee,Person_0249,1.0,DELIVERY CHARGE,238378.38
1879,Main Street Coffee,Person_0249,1.0,"LOTUS SAUCE,(R)",0.00
1880,Main Street Coffee,Person_0249,1.0,LOTUS SPREAD CHIMNEY.,0.00


In [82]:
sales_detail_clean.shape

(1795, 5)

In [83]:
sales_detail_clean.describe()

,Qty,Price
count,1795.000000,1.795000e+03
mean,1.001114,2.521865e+05
std,0.047206,4.513565e+05
min,1.000000,0.000000e+00
25%,1.000000,0.000000e+00
50%,1.000000,0.000000e+00
75%,1.000000,2.383784e+05
max,3.000000,3.933243e+06


In [84]:
sales_detail_clean.isna().sum()

Branch      0
Customer    0
Qty         0
Item        0
Price       0
dtype: int64

In [85]:
sales_detail_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1795 entries, 1 to 1880
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Branch    1795 non-null   object 
 1   Customer  1795 non-null   object 
 2   Qty       1795 non-null   float64
 3   Item      1795 non-null   object 
 4   Price     1795 non-null   float64
dtypes: float64(2), object(3)
memory usage: 84.1+ KB


In [86]:
sales_detail_clean.duplicated().sum()

np.int64(632)

In [87]:
sales_detail_clean[sales_detail_clean.duplicated()]

,Branch,Customer,Qty,Item,Price
17,Conut - Tyre,Person_0131,1.0,[CHOCOLATE DRESSING],0.00
22,Conut - Tyre,Person_0131,1.0,FULL FAT MILK,0.00
24,Conut - Tyre,Person_0131,1.0,MOCHA FRAPPE,536351.34
30,Conut - Tyre,Person_0131,1.0,REGULAR.,0.00
34,Conut - Tyre,Person_0131,1.0,WHIPPED CREAM...,0.00
...,...,...,...,...,...
1855,Main Street Coffee,Person_0133,1.0,"STRAWBERRY,(R)",0.00
1859,Main Street Coffee,Person_0247,1.0,CHIMNEY THE ONE,1251486.48
1861,Main Street Coffee,Person_0247,1.0,CRUSHED LOTUS .(R),0.00
1864,Main Street Coffee,Person_0247,1.0,"LOTUS SAUCE,(R)",0.00


these are actually not duplicate data, they are multiple quantities of the same item in a single order.

### Clean division_summary dataframe

In [39]:
def clean_division_summary(df):
    df_clean = df.copy()
    df_clean['Branch'] = df_clean[0].where(df_clean[3].isna() & df_clean[0].notna())

    df_clean.loc[df_clean['Branch'].str.contains('Summary|Page|30-Jan', na=False), 'Branch'] = None
    df_clean['Branch'] = df_clean['Branch'].ffill()
    
    df_clean = df_clean[df_clean[1].notna() & df_clean[7].notna()]

    df_clean = df_clean[~df_clean[1].str.contains('TOTAL', na=False)]
    
    df_clean = df_clean.rename(columns={1: 'Division', 3: 'Delivery', 4: 'Table', 6: 'TakeAway', 7: 'Total'})
    return df_clean[['Branch', 'Division', 'Delivery', 'Table', 'TakeAway', 'Total']]

division_summary_clean = clean_division_summary(division_summary)

In [41]:
division_summary_clean.head()

,Branch,Division,Delivery,Table,TakeAway,Total
4,Conut - Tyre,Bev Add-ons,0.00,1197189.17,0.00,1197189.17
5,Conut - Tyre,CHIMNEY TOPPINGS,0.00,73241755.78,3933243.19,77174998.97
6,Conut - Tyre,CONUT''S FAVORITE,0.00,4978135.13,0.00,4978135.13
7,Conut - Tyre,Conuts,0.00,0.00,0.00,0.00
8,Conut - Tyre,DRINK TYPE,0.00,0.00,0.00,0.00


In [42]:
division_summary_clean.tail()

,Branch,Division,Delivery,Table,TakeAway,Total
25,Conut - Tyre,MARSHMALLOW OPTIONS,0.00,238378.38,0.00,238378.38
26,Conut - Tyre,MILK OPTIONS,0.00,0.00,0.00,0.00
27,Conut - Tyre,MINI/CONUT/BOWL,0.00,13023405.17,360216.21,13383621.38
28,Conut - Tyre,Shakes,0.00,45768648.79,1430270.28,47198919.07
29,Conut - Tyre,coffee type,0.00,0.00,0.00,0.00


In [45]:
division_summary_clean.shape

(26, 6)

In [70]:
division_summary_clean.describe()

,Branch,Division,Delivery,Table,TakeAway,Total
count,26,26,26,26,26,26
unique,1,26,3,22,15,22
top,Conut - Tyre,Bev Add-ons,0.00,0.00,0.00,0.00
freq,26,1,24,5,10,5


In [56]:
division_summary_clean.isna().sum()

Branch      0
Division    0
Delivery    0
Table       0
TakeAway    0
Total       0
dtype: int64

=> no missing values found

In [50]:
division_summary_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26 entries, 4 to 29
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Branch    26 non-null     object
 1   Division  26 non-null     object
 2   Delivery  26 non-null     object
 3   Table     26 non-null     object
 4   TakeAway  26 non-null     object
 5   Total     26 non-null     object
dtypes: object(6)
memory usage: 1.4+ KB


In [55]:
division_summary_clean[division_summary_clean.duplicated()]

,Branch,Division,Delivery,Table,TakeAway,Total


=> No duplicates found

### clean customer_orders dataframe

In [103]:
def clean_customer_orders_df(df_input):
    df_clean = df_input[df_input[0].astype(str).str.startswith('Person_', na=False)].copy()

    cols_to_keep = [0, 2, 3, 5, 7, 8]
    df_clean = df_clean[cols_to_keep]
 
    df_clean.columns = [
        'Customer_Name', 
        'Phone', 
        'First_Order', 
        'Last_Order', 
        'Total_Spent', 
        'Order_Count'
    ]

    df_clean['Total_Spent'] = (df_clean['Total_Spent']
                               .astype(str)
                               .str.replace(',', '')
                               .str.replace('"', '')
                               .astype(float))
    
    df_clean['Order_Count'] = pd.to_numeric(df_clean['Order_Count'], errors='coerce')
    
    return df_clean


customer_orders_clean = clean_customer_orders_df(customer_orders)


In [104]:
customer_orders_clean.head()

,Customer_Name,Phone,First_Order,Last_Order,Total_Spent,Order_Count
5,Person_0662,27985524,2025-12-31 19:04:,2025-12-31 19:04:,2116800.0,1
6,Person_0663,14060293,2025-12-30 20:49:,2025-12-30 20:49:,3836700.0,1
7,Person_0664,33346800,2025-12-30 19:30:,2025-12-30 19:30:,1256850.0,1
8,Person_0665,72290349,2025-12-29 21:10:,2025-12-29 21:10:,2282910.0,1
9,Person_0666,44665743,2025-12-24 13:33:,2025-12-24 22:52:,0.0,2


In [105]:
customer_orders_clean.tail()

,Customer_Name,Phone,First_Order,Last_Order,Total_Spent,Order_Count
538,Person_1145,94657967,2025-09-24 19:18:,2025-12-11 20:10:,6085799.9,2
539,Person_1146,84271147,2025-09-15 14:34:,2025-09-15 14:34:,2116800.0,1
540,Person_1147,73201017,2025-09-05 14:38:,2025-09-06 09:47:,1918349.9,3
541,Person_1148,02695249,2025-08-27 18:19:,2025-08-29 20:41:,3307499.9,2
542,Person_1149,91187734,2025-10-21 21:30:,2025-10-21 21:30:,2646000.0,1


In [106]:
customer_orders_clean.describe()

,Total_Spent,Order_Count
count,5.080000e+02,508.000000
mean,2.796865e+06,1.216535
std,1.743355e+06,0.628359
min,0.000000e+00,1.000000
25%,1.653750e+06,1.000000
50%,2.415210e+06,1.000000
75%,3.374017e+06,1.000000
max,1.997730e+07,6.000000


In [107]:
customer_orders_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 508 entries, 5 to 542
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Customer_Name  508 non-null    object 
 1   Phone          508 non-null    object 
 2   First_Order    508 non-null    object 
 3   Last_Order     508 non-null    object 
 4   Total_Spent    508 non-null    float64
 5   Order_Count    508 non-null    int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 27.8+ KB


In [108]:
customer_orders_clean[customer_orders_clean.duplicated()]

,Customer_Name,Phone,First_Order,Last_Order,Total_Spent,Order_Count


In [109]:
customer_orders_clean.isna().sum()

Customer_Name    0
Phone            0
First_Order      0
Last_Order       0
Total_Spent      0
Order_Count      0
dtype: int64

### Clean sales_by_item_group

In [112]:
def clean_item_group_sales(df_input):
    clean_data = []
    current_branch = "Unknown"
    current_division = "Unknown"
    current_group = "Unknown"

    for _, row in df_input.iterrows():

        col0 = str(row[0]).strip() if pd.notna(row[0]) else ""
        qty = row[2]
        total_amt = str(row[3]) if pd.notna(row[3]) else "0"

        # 1. Track Hierarchy
        if "Branch:" in col0:
            current_branch = col0.replace("Branch:", "").strip()
            continue
        if "Division:" in col0:
            current_division = col0.replace("Division:", "").strip()
            continue
        if "Group:" in col0:
            current_group = col0.replace("Group:", "").strip()
            continue
 
        try:
            qty_val = float(qty)
            # Skip rows that are totals/summaries or empty
            if "Total" in col0 or col0 == "" or np.isnan(qty_val):
                continue
            
            # 3. Clean Price/Amount
            amount = float(total_amt.replace(',', '').replace('"', ''))
            
            clean_data.append({
                'Branch': current_branch,
                'Division': current_division,
                'Group': current_group,
                'Item': col0,
                'Qty': qty_val,
                'Total_Amount': amount
            })
        except (ValueError, TypeError):
            continue

    return pd.DataFrame(clean_data)


sales_items_clean = clean_item_group_sales(sales_by_item_group)



In [113]:
sales_items_clean.head()

,Branch,Division,Group,Item,Qty,Total_Amount
0,Conut - Tyre,Hot-Coffee Based,Hot-Coffee Based,AMERICAN COFFEE,8.0,2860540.50
1,Conut - Tyre,Hot-Coffee Based,Hot-Coffee Based,CAPPUCCINO,1.0,417162.16
2,Conut - Tyre,Hot-Coffee Based,Hot-Coffee Based,CAFFE LATTE,47.0,19606621.36
3,Conut - Tyre,Hot-Coffee Based,Hot-Coffee Based,CAFE MOCHA,5.0,2536081.12
4,Conut - Tyre,Hot-Coffee Based,Hot-Coffee Based,CARAMEL MACHIATO,7.0,3754459.41


In [114]:
sales_items_clean.tail()

,Branch,Division,Group,Item,Qty,Total_Amount
1153,Main Street Coffee,MARSHMALLOW OPTIONS,MARSHMALLOW OPTIONS,NO MARSHMALLLOWS,53.0,0.00
1154,Main Street Coffee,CONUT''S FAVORITE,CONUT''S FAVORITE,HOT CHOCOLATE,71.0,33849729.28
1155,Main Street Coffee,CONUT''S FAVORITE,CONUT''S FAVORITE,MATCHA LATTE,3.0,1787837.86
1156,Main Street Coffee,CONUT''S FAVORITE,CONUT''S FAVORITE,AFFOGATO,6.0,3218108.07
1157,Main Street Coffee,CONUT''S FAVORITE,CONUT''S FAVORITE,PISTACHIO LATTE,3.0,1787837.86


In [115]:
sales_items_clean.describe()

,Qty,Total_Amount
count,1158.000000,1.158000e+03
mean,80.267703,1.804733e+07
std,192.362739,7.845656e+07
min,0.000000,0.000000e+00
25%,3.000000,0.000000e+00
50%,13.000000,2.383784e+05
75%,58.000000,2.145405e+06
max,2181.000000,1.088793e+09


In [116]:
sales_items_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1158 entries, 0 to 1157
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Branch        1158 non-null   object 
 1   Division      1158 non-null   object 
 2   Group         1158 non-null   object 
 3   Item          1158 non-null   object 
 4   Qty           1158 non-null   float64
 5   Total_Amount  1158 non-null   float64
dtypes: float64(2), object(4)
memory usage: 54.4+ KB


In [117]:
sales_items_clean.isna().sum()

Branch          0
Division        0
Group           0
Item            0
Qty             0
Total_Amount    0
dtype: int64

In [118]:
sales_items_clean[sales_items_clean.duplicated()]

,Branch,Division,Group,Item,Qty,Total_Amount


### Clean branch_tax_summary

In [121]:
def clean_tax_summary(df_input):
    clean_data = []
    current_branch = "Unknown"

    for _, row in df_input.iterrows():
        col0 = str(row[0]).strip() if pd.notna(row[0]) else ""

        if "Branch Name:" in col0:
            current_branch = col0.replace("Branch Name:", "").strip()
            continue

        if "Total By Branch" in col0:
            try:
                vat_11 = float(str(row[1]).replace(',', '').replace('"', ''))
                total_revenue = float(str(row[8]).replace(',', '').replace('"', ''))
                
                clean_data.append({
                    'Branch': current_branch,
                    'VAT_11_Percent': vat_11,
                    'Total_Revenue': total_revenue
                })
            except (ValueError, TypeError):
                continue

    return pd.DataFrame(clean_data)

branch_revenue_clean = clean_tax_summary(branch_tax_summary)

In [122]:
branch_revenue_clean.head()

,Branch,VAT_11_Percent,Total_Revenue
0,Conut,4.270485e+08,4.270485e+08
1,Conut - Tyre,5.631346e+08,5.631346e+08
2,Conut Jnah,6.235977e+08,6.235977e+08
3,Main Street Coffee,5.798939e+08,5.798939e+08


In [123]:
branch_revenue_clean.tail()

,Branch,VAT_11_Percent,Total_Revenue
0,Conut,4.270485e+08,4.270485e+08
1,Conut - Tyre,5.631346e+08,5.631346e+08
2,Conut Jnah,6.235977e+08,6.235977e+08
3,Main Street Coffee,5.798939e+08,5.798939e+08


In [124]:
branch_revenue_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Branch          4 non-null      object 
 1   VAT_11_Percent  4 non-null      float64
 2   Total_Revenue   4 non-null      float64
dtypes: float64(2), object(1)
memory usage: 228.0+ bytes


In [125]:
branch_revenue_clean.describe()

,VAT_11_Percent,Total_Revenue
count,4.000000e+00,4.000000e+00
mean,5.484187e+08,5.484187e+08
std,8.483285e+07,8.483285e+07
min,4.270485e+08,4.270485e+08
25%,5.291131e+08,5.291131e+08
50%,5.715142e+08,5.715142e+08
75%,5.908198e+08,5.908198e+08
max,6.235977e+08,6.235977e+08


In [126]:
branch_revenue_clean.isna().sum()

Branch            0
VAT_11_Percent    0
Total_Revenue     0
dtype: int64

In [127]:
branch_revenue_clean[branch_revenue_clean.duplicated()]

,Branch,VAT_11_Percent,Total_Revenue


### Clean monthly_branch_sales

In [158]:
def clean_monthly_sales_df(df_input):
    clean_data = []
    current_branch = "Unknown"
    
    months_list = [
        'January', 'February', 'March', 'April', 'May', 'June', 
        'July', 'August', 'September', 'October', 'November', 'December'
    ]

    for _, row in df_input.iterrows():
        col0 = str(row[0]).strip() if pd.notna(row[0]) else ""
        
        if "Branch Name:" in col0:
            current_branch = col0.replace("Branch Name:", "").strip()
            continue

        if col0 in months_list:
            try:
                year = row[2]
                sales_str = str(row[3])
                sales_val = float(sales_str.replace(',', '').replace('"', ''))
                
                clean_data.append({
                    'Branch': current_branch,
                    'Month': col0,
                    'Year': int(float(year)) if pd.notna(year) else 2025,
                    'Total_Sales': sales_val
                })
            except (ValueError, TypeError):
                continue

    df_result = pd.DataFrame(clean_data)

    df_result['Date'] = pd.to_datetime(
        df_result['Month'] + ' ' + df_result['Year'].astype(str)
    )
    
    return df_result.sort_values(['Branch', 'Date'])

monthly_sales_clean = clean_monthly_sales_df(monthly_branch_sales)


C:\Users\hsake\AppData\Local\Temp\ipykernel_31144\2099190968.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_result['Date'] = pd.to_datetime(


In [131]:
monthly_sales_clean.head()

,Branch,Month,Year,Total_Sales,Date
0,Conut,August,2025,5.540748e+08,2025-08-01
1,Conut,September,2025,7.843854e+08,2025-09-01
2,Conut,October,2025,1.137352e+09,2025-10-01
3,Conut,November,2025,1.351166e+09,2025-11-01
4,Conut,December,2025,6.788751e+07,2025-12-01


In [132]:
monthly_sales_clean.tail()

,Branch,Month,Year,Total_Sales,Date
14,Conut Jnah,December,2025,2.878191e+09,2025-12-01
15,Main Street Coffee,September,2025,1.458425e+08,2025-09-01
16,Main Street Coffee,October,2025,9.205882e+08,2025-10-01
17,Main Street Coffee,November,2025,1.171534e+09,2025-11-01
18,Main Street Coffee,December,2025,3.074216e+09,2025-12-01


In [133]:
monthly_sales_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Branch       19 non-null     object        
 1   Month        19 non-null     object        
 2   Year         19 non-null     int64         
 3   Total_Sales  19 non-null     float64       
 4   Date         19 non-null     datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1), object(2)
memory usage: 892.0+ bytes


In [134]:
monthly_sales_clean.describe()

,Year,Total_Sales,Date
count,19.0,1.900000e+01,19
mean,2025.0,1.056488e+09,2025-10-04 10:06:18.947368448
min,2025.0,6.788751e+07,2025-08-01 00:00:00
25%,2025.0,5.158051e+08,2025-09-01 00:00:00
50%,2025.0,9.205882e+08,2025-10-01 00:00:00
75%,2025.0,1.154443e+09,2025-11-01 00:00:00
max,2025.0,3.074216e+09,2025-12-01 00:00:00
std,0.0,8.211380e+08,NaN


In [135]:
monthly_sales_clean.isna().sum()

Branch         0
Month          0
Year           0
Total_Sales    0
Date           0
dtype: int64

In [136]:
monthly_sales_clean[monthly_sales_clean.duplicated()]

,Branch,Month,Year,Total_Sales,Date


In [ ]:
### Clean avg_sales_menu

In [139]:
def clean_avg_menu_sales(df_input):
    clean_data = []
    current_branch = "Unknown"
    menu_types = ['DELIVERY', 'TABLE', 'TAKE AWAY']

    for _, row in df_input.iterrows():
        col0 = str(row[0]).strip() if pd.notna(row[0]) else ""

        if col0 and not any(m in col0 for m in menu_types) and "Total" not in col0 and "Menu" not in col0 and "30-Jan" not in col0:
            if len(col0) > 3: 
                current_branch = col0
            continue

        if col0 in menu_types:
            try:
                num_customers = float(str(row[1]).replace(',', ''))
                total_sales = float(str(row[2]).replace(',', ''))
                avg_per_cust = float(str(row[3]).replace(',', ''))
                
                clean_data.append({
                    'Branch': current_branch,
                    'Menu_Type': col0,
                    'Customer_Count': num_customers,
                    'Total_Sales': total_sales,
                    'Avg_Spent_Per_Customer': avg_per_cust
                })
            except (ValueError, TypeError):
                continue

    return pd.DataFrame(clean_data)

avg_sales_menu_clean = clean_avg_menu_sales(avg_sales_menu_v1)

In [140]:
avg_sales_menu_clean.head()

,Branch,Menu_Type,Customer_Count,Total_Sales,Avg_Spent_Per_Customer
0,Conut - Tyre,DELIVERY,79.0,1.969787e+08,2493400.96
1,Conut - Tyre,TAKE AWAY,3038.0,4.921979e+09,1620138.08
2,Conut,DELIVERY,6.0,9.745703e+06,1624283.79
3,Conut,TABLE,2609.0,3.679878e+09,1410455.40
4,Conut,TAKE AWAY,129.0,1.926356e+08,1493298.87


In [141]:
avg_sales_menu_clean.tail()

,Branch,Menu_Type,Customer_Count,Total_Sales,Avg_Spent_Per_Customer
2,Conut,DELIVERY,6.0,9.745703e+06,1624283.79
3,Conut,TABLE,2609.0,3.679878e+09,1410455.40
4,Conut,TAKE AWAY,129.0,1.926356e+08,1493298.87
5,Conut Jnah,TABLE,5045.0,5.669070e+09,1123700.62
6,Main Street Coffee,TABLE,3640.0,5.271762e+09,1448286.39


In [142]:
avg_sales_menu_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Branch                  7 non-null      object 
 1   Menu_Type               7 non-null      object 
 2   Customer_Count          7 non-null      float64
 3   Total_Sales             7 non-null      float64
 4   Avg_Spent_Per_Customer  7 non-null      float64
dtypes: float64(3), object(2)
memory usage: 412.0+ bytes


In [143]:
avg_sales_menu_clean.describe()

,Customer_Count,Total_Sales,Avg_Spent_Per_Customer
count,7.000000,7.000000e+00,7.000000e+00
mean,2078.000000,2.848864e+09,1.601938e+06
std,2022.139131,2.612822e+09,4.274585e+05
min,6.000000,9.745703e+06,1.123701e+06
25%,104.000000,1.948071e+08,1.429371e+06
50%,2609.000000,3.679878e+09,1.493299e+06
75%,3339.000000,5.096871e+09,1.622211e+06
max,5045.000000,5.669070e+09,2.493401e+06


In [144]:
avg_sales_menu_clean[avg_sales_menu_clean.duplicated()]

,Branch,Menu_Type,Customer_Count,Total_Sales,Avg_Spent_Per_Customer


In [145]:
avg_sales_menu_clean.isna().sum()

Branch                    0
Menu_Type                 0
Customer_Count            0
Total_Sales               0
Avg_Spent_Per_Customer    0
dtype: int64

### Clean attendance_logs

In [147]:
import pandas as pd
import numpy as np

def clean_attendance_df(df_input):
    clean_data = []
    current_branch = "Unknown"
    current_emp_name = "Unknown"
    current_emp_id = "Unknown"

    for _, row in df_input.iterrows():
        col0 = str(row[0]).strip() if pd.notna(row[0]) else ""
        col2 = str(row[2]).strip() if pd.notna(row[2]) else ""
        col5 = str(row[5]).strip() if pd.notna(row[5]) else ""

        if col0 == "" and col2 == "" and col5 == "" and pd.notna(row[1]):
            val = str(row[1]).strip()
            if "30-Jan" not in val and "PUNCH" not in val:
                current_branch = val
            continue

        if "NAME :" in col2:
            current_emp_name = col2.split("NAME :")[-1].strip()
            current_emp_id = str(row[1]).replace("EMP ID :", "").strip()
            continue

        if len(col0) == 9 and col0[2] == '-' and col0[6] == '-':
            try:

                h, m, s = map(int, col5.split('.'))
                decimal_hours = h + (m / 60) + (s / 3600)
                
                clean_data.append({
                    'Branch': current_branch,
                    'Employee_ID': current_emp_id,
                    'Employee_Name': current_emp_name,
                    'Date': col0,
                    'Punch_In': col2,
                    'Punch_Out': str(row[4]),
                    'Work_Hours': round(decimal_hours, 2)
                })
            except:
                continue

    df_result = pd.DataFrame(clean_data)
    df_result['Date'] = pd.to_datetime(df_result['Date'], format='%d-%b-%y')
    
    return df_result

attendance_clean = clean_attendance_df(attendance_logs)


In [148]:
attendance_clean.head()

,Branch,Employee_ID,Employee_Name,Date,Punch_In,Punch_Out,Work_Hours
0,Main Street Coffee,1.0,Person_0001,2025-12-01,07.39.35,19.37.56,11.97
1,Main Street Coffee,1.0,Person_0001,2025-12-02,15.14.59,23.51.33,8.61
2,Main Street Coffee,1.0,Person_0001,2025-12-03,15.12.21,00.17.12,9.08
3,Main Street Coffee,1.0,Person_0001,2025-12-06,16.03.47,00.27.15,8.39
4,Main Street Coffee,1.0,Person_0001,2025-12-07,13.15.46,23.43.34,10.46


In [149]:
attendance_clean.tail()

,Branch,Employee_ID,Employee_Name,Date,Punch_In,Punch_Out,Work_Hours
306,Conut Jnah,54.0,Person_0016,2025-12-24,08.57.18,18.00.09,9.05
307,Conut Jnah,54.0,Person_0016,2025-12-26,09.07.09,13.16.38,4.16
308,Conut Jnah,54.0,Person_0016,2025-12-27,13.16.44,22.04.50,8.80
309,Conut Jnah,54.0,Person_0016,2025-12-28,12.57.10,22.05.27,9.14
310,Conut Jnah,54.0,Person_0016,2025-12-29,09.02.44,18.01.11,8.97


In [150]:
attendance_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311 entries, 0 to 310
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Branch         311 non-null    object        
 1   Employee_ID    311 non-null    object        
 2   Employee_Name  311 non-null    object        
 3   Date           311 non-null    datetime64[ns]
 4   Punch_In       311 non-null    object        
 5   Punch_Out      311 non-null    object        
 6   Work_Hours     311 non-null    float64       
dtypes: datetime64[ns](1), float64(1), object(5)
memory usage: 17.1+ KB


In [151]:
attendance_clean.describe()

,Date,Work_Hours
count,311,311.00000
mean,2025-12-15 07:56:54.790996736,7.43717
min,2025-12-01 00:00:00,0.00000
25%,2025-12-08 00:00:00,6.11500
50%,2025-12-16 00:00:00,8.50000
75%,2025-12-23 00:00:00,9.10000
max,2025-12-29 00:00:00,23.97000
std,NaN,4.10354


In [152]:
attendance_clean.isna().sum()

Branch           0
Employee_ID      0
Employee_Name    0
Date             0
Punch_In         0
Punch_Out        0
Work_Hours       0
dtype: int64

In [153]:
attendance_clean[attendance_clean.duplicated()]

,Branch,Employee_ID,Employee_Name,Date,Punch_In,Punch_Out,Work_Hours


### Export the cleaned files

In [157]:
export_path = r'C:\Users\Public\Hackathon\Conut bakery Scaled Data_\prepared data'

if not os.path.exists(export_path):
    os.makedirs(export_path)
    print(f"Created directory: {export_path}")

exports = {
    'cleaned_sales_detail.csv': sales_detail_clean,
    'cleaned_customer_orders.csv': customer_orders_clean,
    'cleaned_sales_by_item_group.csv': sales_items_clean,
    'cleaned_branch_tax_summary.csv': branch_revenue_clean,
    'cleaned_monthly_branch_sales.csv': monthly_sales_clean,
    'cleaned_avg_sales_menu.csv': avg_sales_menu_clean,
    'cleaned_attendance_logs.csv': attendance_clean,
    'division_summary_clean.csv': division_summary_clean
}


for filename, df in exports.items():
    full_file_path = os.path.join(export_path, filename)
    df.to_csv(full_file_path, index=False, encoding='utf-8-sig')

